# Large Language Model Benchmarking Tutorial

This notebook provides a comprehensive guide to benchmarking Large Language Models (LLMs) using the tools in this repository.

## Table of Contents
1. [Setup and Installation](#setup)
2. [Understanding Benchmarking Metrics](#metrics)
3. [Zero-shot Testing](#zero-shot)
4. [Few-shot Testing](#few-shot)
5. [Model Comparison](#comparison)
6. [Results Analysis](#analysis)
7. [Custom Benchmarks](#custom)

---

## 1. Setup and Installation {#setup}

First, let's install the required packages and import necessary libraries.

In [ ]:
# Install required packages if not already installed
import subprocess
import sys

def install_package(package):
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

# Install core packages
packages = [
    "torch",
    "transformers",
    "datasets",
    "scikit-learn",
    "numpy",
    "pandas",
    "matplotlib",
    "seaborn",
    "tqdm"
]

for package in packages:
    install_package(package)

print("✅ All packages installed successfully!")

In [ ]:
# Import all necessary libraries
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from datasets import Dataset
import time
import json
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"Using CUDA: {torch.cuda.is_available()}")

## 2. Understanding Benchmarking Metrics {#metrics}

Before we start benchmarking, let's understand the key metrics we'll be using:

In [ ]:
class BenchmarkMetrics:
    """Class to calculate and explain benchmarking metrics"""
    
    @staticmethod
    def accuracy(y_true, y_pred):
        """Calculate accuracy: (TP + TN) / (TP + TN + FP + FN)"""
        return accuracy_score(y_true, y_pred)
    
    @staticmethod
    def f1(y_true, y_pred, average='weighted'):
        """Calculate F1-score: 2 * (precision * recall) / (precision + recall)"""
        return f1_score(y_true, y_pred, average=average)
    
    @staticmethod
    def precision(y_true, y_pred, average='weighted'):
        """Calculate precision: TP / (TP + FP)"""
        return precision_score(y_true, y_pred, average=average)
    
    @staticmethod
    def recall(y_true, y_pred, average='weighted'):
        """Calculate recall: TP / (TP + FN)"""
        return recall_score(y_true, y_pred, average=average)
    
    @staticmethod
    def exact_match(y_true, y_pred):
        """Calculate exact match accuracy for text generation tasks"""
        matches = sum(1 for true, pred in zip(y_true, y_pred) if true.strip().lower() == pred.strip().lower())
        return matches / len(y_true)
    
    @staticmethod
    def inference_time(func, inputs, iterations=10):
        """Measure average inference time"""
        times = []
        for _ in range(iterations):
            start = time.time()
            func(inputs)
            end = time.time()
            times.append(end - start)
        return np.mean(times) * 1000  # Return in milliseconds

# Display metric explanations
metrics_explanation = pd.DataFrame({
    'Metric': ['Accuracy', 'F1-Score', 'Precision', 'Recall', 'Exact Match', 'Inference Time'],
    'Formula': ['(TP + TN) / Total', '2 * (P * R) / (P + R)', 'TP / (TP + FP)', 'TP / (TP + FN)', 'Exact matches / Total', 'Average time per prediction'],
    'Best Value': ['1.0 (100%)', '1.0 (100%)', '1.0 (100%)', '1.0 (100%)', '1.0 (100%)', 'Lower is better'],
    'Use Case': ['Overall performance', 'Balanced precision/recall', 'Avoiding false positives', 'Finding all positives', 'Text generation accuracy', 'Speed assessment']
})

print("Benchmarking Metrics Overview:")
print(metrics_explanation.to_string(index=False))

## 3. Zero-shot Testing {#zero-shot}

Zero-shot testing evaluates how well models perform on tasks without any task-specific training examples.

In [ ]:
class LLMBenchmarkSuite:
    """Comprehensive LLM benchmarking suite"""
    
    def __init__(self):
        self.models = {
            'bert': 'bert-base-uncased',
            'distilbert': 'distilbert-base-uncased',
            'albert': 'albert-base-v2'
        }
        self.results = {}
        self.metrics = BenchmarkMetrics()
    
    def load_model(self, model_name):
        """Load a model and tokenizer"""
        print(f"Loading {model_name}...")
        model_id = self.models[model_name]
        
        # Create a text classification pipeline
        classifier = pipeline(
            "text-classification",
            model=model_id,
            tokenizer=model_id,
            return_all_scores=True
        )
        print(f"✅ {model_name.upper()} loaded successfully!")
        return classifier
    
    def create_sample_data(self):
        """Create sample datasets for benchmarking"""
        # Sentiment Analysis Dataset
        sentiment_data = {
            'text': [
                "I love this product! It's amazing.",
                "This is the worst thing I've ever bought.",
                "The item is okay, nothing special.",
                "Excellent quality and fast shipping!",
                "Terrible customer service, very disappointed.",
                "Good value for money, satisfied with purchase.",
                "Not what I expected, poor quality materials.",
                "Outstanding product, exceeded my expectations!"
            ],
            'label': [1, 0, 1, 1, 0, 1, 0, 1]  # 1: Positive, 0: Negative
        }
        
        # Question-Answering Dataset
        qa_data = {
            'question': [
                "What is the capital of France?",
                "How many days are in a week?",
                "What is 2 + 2?",
                "Who wrote Romeo and Juliet?",
                "What is the largest planet in our solar system?"
            ],
            'answer': [
                "Paris",
                "Seven",
                "Four", 
                "William Shakespeare",
                "Jupiter"
            ]
        }
        
        # Mathematical Reasoning Dataset
        math_data = {
            'problem': [
                "If a train travels 60 miles per hour for 3 hours, how far does it travel?",
                "A rectangle has length 8 and width 5. What is its area?",
                "What is 15% of 200?",
                "If you have 24 apples and eat 1/3 of them, how many are left?",
                "A circle has radius 5. What is its circumference? (Use π ≈ 3.14)"
            ],
            'solution': [
                "180",
                "40", 
                "30",
                "16",
                "31.4"
            ]
        }
        
        return {
            'sentiment': pd.DataFrame(sentiment_data),
            'qa': pd.DataFrame(qa_data),
            'math': pd.DataFrame(math_data)
        }

# Create benchmark suite and sample data
benchmark = LLMBenchmarkSuite()
datasets = benchmark.create_sample_data()

print("📋 Sample Datasets Created:")
for name, df in datasets.items():
    print(f"  • {name.title()}: {len(df)} examples")
    print(f"    Preview: {df.iloc[0].to_dict()}")
    print()

In [ ]:
def benchmark_sentiment_analysis(model_name):
    """Benchmark sentiment analysis performance"""
    print(f"\nBenchmarking Sentiment Analysis - {model_name.upper()}")
    
    # Load model
    classifier = benchmark.load_model(model_name)
    
    # Get data
    data = datasets['sentiment']
    texts = data['text'].tolist()
    true_labels = data['label'].tolist()
    
    # Make predictions and measure time
    start_time = time.time()
    predictions = []
    
    for text in tqdm(texts, desc="Processing"):
        result = classifier(text)
        # Convert to binary classification (positive/negative)
        pred_label = 1 if result[0]['label'] == 'POSITIVE' or result[0]['score'] > 0.5 else 0
        predictions.append(pred_label)
    
    inference_time = (time.time() - start_time) * 1000 / len(texts)
    
    # Calculate metrics
    accuracy = benchmark.metrics.accuracy(true_labels, predictions)
    f1 = benchmark.metrics.f1(true_labels, predictions)
    precision = benchmark.metrics.precision(true_labels, predictions)
    recall = benchmark.metrics.recall(true_labels, predictions)
    
    results = {
        'accuracy': accuracy,
        'f1_score': f1,
        'precision': precision,
        'recall': recall,
        'inference_time_ms': inference_time
    }
    
    print(f"\n📊 Results for {model_name.upper()}:")
    for metric, value in results.items():
        if 'time' in metric:
            print(f"  • {metric.replace('_', ' ').title()}: {value:.2f} ms")
        else:
            print(f"  • {metric.replace('_', ' ').title()}: {value:.3f}")
    
    return results

# Test sentiment analysis with one model
distilbert_results = benchmark_sentiment_analysis('distilbert')

## 4. Few-shot Testing {#few-shot}

Few-shot testing provides the model with a few examples before asking it to make predictions.

In [ ]:
def few_shot_prompting_example():
    """Demonstrate few-shot prompting for text classification"""
    
    print("Few-shot Learning Example")
    print("=" * 50)
    
    # Example of few-shot prompt structure
    few_shot_prompt = """
Classify the sentiment of the following texts as POSITIVE or NEGATIVE:

Examples:
Text: "I love this product! It's amazing."
Sentiment: POSITIVE

Text: "This is the worst thing I've ever bought."
Sentiment: NEGATIVE

Text: "Excellent quality and fast shipping!"
Sentiment: POSITIVE

Now classify:
Text: "The item is okay, nothing special."
Sentiment: """
    
    print("Few-shot Prompt Structure:")
    print(few_shot_prompt)
    
    # Simulate few-shot learning benefits
    zero_shot_accuracy = 0.75
    few_shot_accuracy = 0.85
    improvement = ((few_shot_accuracy - zero_shot_accuracy) / zero_shot_accuracy) * 100
    
    print(f"\nExpected Performance Improvement:")
    print(f"  • Zero-shot accuracy: {zero_shot_accuracy:.1%}")
    print(f"  • Few-shot accuracy: {few_shot_accuracy:.1%}")
    print(f"  • Improvement: +{improvement:.1f}%")
    
    return {
        'zero_shot': zero_shot_accuracy,
        'few_shot': few_shot_accuracy,
        'improvement_percent': improvement
    }

few_shot_results = few_shot_prompting_example()

## 5. Model Comparison {#comparison}

Let's compare different models across multiple tasks and create visualizations.

In [ ]:
def run_comprehensive_benchmark():
    """Run benchmarks on multiple models and tasks"""
    
    # Simulated results for demonstration (in real scenario, you'd run actual benchmarks)
    benchmark_results = {
        'BERT': {
            'sentiment_accuracy': 0.85,
            'qa_exact_match': 0.80,
            'math_accuracy': 0.25,
            'inference_time_ms': 150
        },
        'DistilBERT': {
            'sentiment_accuracy': 0.82,
            'qa_exact_match': 0.75,
            'math_accuracy': 0.22,
            'inference_time_ms': 85
        },
        'ALBERT': {
            'sentiment_accuracy': 0.83,
            'qa_exact_match': 0.78,
            'math_accuracy': 0.28,
            'inference_time_ms': 120
        }
    }
    
    # Convert to DataFrame for easier handling
    df_results = pd.DataFrame(benchmark_results).T
    
    print("Comprehensive Benchmark Results:")
    print(df_results.round(3))
    
    return df_results

# Run comprehensive benchmark
results_df = run_comprehensive_benchmark()

In [ ]:
def create_benchmark_visualizations(results_df):
    """Create comprehensive visualizations of benchmark results"""
    
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('LLM Benchmark Results Comparison', fontsize=16, fontweight='bold')
    
    # 1. Accuracy Comparison
    accuracy_metrics = ['sentiment_accuracy', 'qa_exact_match', 'math_accuracy']
    accuracy_data = results_df[accuracy_metrics]
    
    accuracy_data.plot(kind='bar', ax=axes[0, 0])
    axes[0, 0].set_title('Task Accuracy Comparison')
    axes[0, 0].set_ylabel('Accuracy Score')
    axes[0, 0].legend(['Sentiment', 'Q&A', 'Math'])
    axes[0, 0].tick_params(axis='x', rotation=45)
    
    # 2. Inference Time Comparison
    results_df['inference_time_ms'].plot(kind='bar', ax=axes[0, 1], color='orange')
    axes[0, 1].set_title('Inference Time Comparison')
    axes[0, 1].set_ylabel('Time (milliseconds)')
    axes[0, 1].tick_params(axis='x', rotation=45)
    
    # 3. Performance vs Speed Trade-off
    avg_accuracy = results_df[accuracy_metrics].mean(axis=1)
    axes[1, 0].scatter(results_df['inference_time_ms'], avg_accuracy, s=100)
    for i, model in enumerate(results_df.index):
        axes[1, 0].annotate(model, 
                           (results_df['inference_time_ms'].iloc[i], avg_accuracy.iloc[i]),
                           xytext=(5, 5), textcoords='offset points')
    axes[1, 0].set_xlabel('Inference Time (ms)')
    axes[1, 0].set_ylabel('Average Accuracy')
    axes[1, 0].set_title('Performance vs Speed Trade-off')
    
    # 4. Radar Chart for Overall Performance
    categories = ['Sentiment\nAccuracy', 'Q&A\nExact Match', 'Math\nAccuracy', 'Speed\n(Inverse)']
    
    # Normalize speed (inverse so higher is better)
    speed_normalized = 1 - (results_df['inference_time_ms'] - results_df['inference_time_ms'].min()) / \
                      (results_df['inference_time_ms'].max() - results_df['inference_time_ms'].min())
    
    angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False)
    angles = np.concatenate((angles, [angles[0]]))  # Complete the circle
    
    ax_radar = plt.subplot(224, projection='polar')
    
    for i, model in enumerate(results_df.index):
        values = [
            results_df.loc[model, 'sentiment_accuracy'],
            results_df.loc[model, 'qa_exact_match'], 
            results_df.loc[model, 'math_accuracy'],
            speed_normalized.iloc[i]
        ]
        values += [values[0]]  # Complete the circle
        
        ax_radar.plot(angles, values, 'o-', linewidth=2, label=model)
        ax_radar.fill(angles, values, alpha=0.25)
    
    ax_radar.set_xticks(angles[:-1])
    ax_radar.set_xticklabels(categories)
    ax_radar.set_ylim(0, 1)
    ax_radar.set_title('Overall Performance Radar')
    ax_radar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
    
    plt.tight_layout()
    plt.show()
    
    return fig

# Create visualizations
visualization_fig = create_benchmark_visualizations(results_df)

## 6. Results Analysis {#analysis}

Let's analyze the benchmark results and provide insights.

In [ ]:
def generate_benchmark_insights(results_df):
    """Generate insights from benchmark results"""
    
    print("Benchmark Analysis & Insights")
    print("=" * 50)
    
    # Best performing model per task
    best_sentiment = results_df['sentiment_accuracy'].idxmax()
    best_qa = results_df['qa_exact_match'].idxmax()
    best_math = results_df['math_accuracy'].idxmax()
    fastest_model = results_df['inference_time_ms'].idxmin()
    
    print(f"Best Performing Models by Task:")
    print(f"  • Sentiment Analysis: {best_sentiment} ({results_df.loc[best_sentiment, 'sentiment_accuracy']:.1%})")
    print(f"  • Question Answering: {best_qa} ({results_df.loc[best_qa, 'qa_exact_match']:.1%})")
    print(f"  • Mathematical Reasoning: {best_math} ({results_df.loc[best_math, 'math_accuracy']:.1%})")
    print(f"  • Fastest Inference: {fastest_model} ({results_df.loc[fastest_model, 'inference_time_ms']:.0f} ms)")
    
    # Overall performance ranking
    accuracy_cols = ['sentiment_accuracy', 'qa_exact_match', 'math_accuracy']
    overall_performance = results_df[accuracy_cols].mean(axis=1).sort_values(ascending=False)
    
    print(f"\nOverall Performance Ranking:")
    for i, (model, score) in enumerate(overall_performance.items(), 1):
        print(f"  {i}. {model}: {score:.1%} average accuracy")
    
    # Performance vs Speed Analysis
    print(f"\nPerformance vs Speed Trade-offs:")
    
    for model in results_df.index:
        avg_acc = results_df.loc[model, accuracy_cols].mean()
        speed = results_df.loc[model, 'inference_time_ms']
        efficiency = avg_acc / (speed / 100)  # Performance per 100ms
        
        print(f"  • {model}:")
        print(f"    - Average Accuracy: {avg_acc:.1%}")
        print(f"    - Inference Time: {speed:.0f} ms")
        print(f"    - Efficiency Score: {efficiency:.3f}")
        print()
    
    # Recommendations
    print(f"Recommendations:")
    
    if fastest_model == overall_performance.index[0]:
        print(f"  • {fastest_model} offers the best balance of accuracy and speed")
    else:
        print(f"  • For highest accuracy: Use {overall_performance.index[0]}")
        print(f"  • For fastest inference: Use {fastest_model}")
    
    if results_df.loc[:, 'math_accuracy'].max() < 0.5:
        print(f"  • Mathematical reasoning is challenging for all models - consider fine-tuning")
    
    print(f"  • Sentiment analysis shows strongest performance across all models")
    print(f"  • Consider few-shot prompting to improve Q&A performance")
    
    return {
        'best_overall': overall_performance.index[0],
        'fastest': fastest_model,
        'overall_ranking': overall_performance.to_dict()
    }

# Generate insights
insights = generate_benchmark_insights(results_df)

## 7. Custom Benchmarks {#custom}

Learn how to create and run your own custom benchmarks.

In [ ]:
def create_custom_benchmark():
    """Template for creating custom benchmarks"""
    
    print("🛠️ Custom Benchmark Template")
    print("=" * 40)
    
    # Step 1: Define your task
    task_description = """
    Step 1: Define Your Task
    
    Example: Email Spam Detection
    - Input: Email text
    - Output: Spam (1) or Not Spam (0)
    - Metric: Accuracy, Precision, Recall
    """
    print(task_description)
    
    # Step 2: Prepare your dataset
    custom_dataset = {
        'email_text': [
            "Congratulations! You've won $1 million! Click here to claim now!",
            "Hi John, can we reschedule our meeting for tomorrow at 2 PM?",
            "URGENT: Your account will be suspended unless you verify immediately!",
            "Thanks for the great presentation today. Looking forward to next steps.",
            "This is not scam, let's run Notebooks and learn together!",
            "Your package delivery is scheduled for tomorrow between 9-11 AM."
        ],
        'is_spam': [1, 0, 1, 0, 1, 0]  # 1 = spam, 0 = not spam
    }
    
    custom_df = pd.DataFrame(custom_dataset)
    
    print("Step 2: Your Custom Dataset")
    print(custom_df)
    
    # Step 3: Define evaluation function
    def evaluate_spam_detection(model, test_data):
        """Custom evaluation function for spam detection"""
        predictions = []
        
        for email in test_data['email_text']:
            # Simulate model prediction (replace with actual model inference)
            spam_indicators = ['win', 'free', 'click', 'urgent', 'viagra', 'claim']
            spam_score = sum(1 for word in spam_indicators if word.lower() in email.lower())
            prediction = 1 if spam_score >= 2 else 0
            predictions.append(prediction)
        
        # Calculate metrics
        accuracy = accuracy_score(test_data['is_spam'], predictions)
        precision = precision_score(test_data['is_spam'], predictions)
        recall = recall_score(test_data['is_spam'], predictions)
        f1 = f1_score(test_data['is_spam'], predictions)
        
        return {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1_score': f1
        }
    
    # Step 4: Run evaluation
    results = evaluate_spam_detection(None, custom_df)  # Model parameter not used in this example
    
    print("\nStep 3: Evaluation Results")
    for metric, value in results.items():
        print(f"  • {metric.title().replace('_', ' ')}: {value:.3f}")
    
    # Step 5: Best practices
    best_practices = """
    Best Practices for Custom Benchmarks:
    
    1. Use diverse, representative datasets
    2. Include edge cases and challenging examples
    3. Choose appropriate metrics for your task
    4. Consider class imbalance in your data
    5. Test multiple models and compare results
    6. Document your benchmark methodology
    7. Make your benchmark reproducible
    8. Validate results with domain experts
    """
    print(best_practices)
    
    return custom_df, results

# Create custom benchmark example
custom_data, custom_results = create_custom_benchmark()

## Summary and Next Steps

This notebook covered the fundamentals of LLM benchmarking:

### What You Learned:
- **Benchmarking Metrics**: Accuracy, F1-score, precision, recall, exact match, inference time
- **Testing Approaches**: Zero-shot, few-shot, and fine-tuned evaluation methods
- **Model Comparison**: Systematic comparison of BERT, DistilBERT, and ALBERT
- **Results Visualization**: Creating comprehensive charts and radar plots
- **Performance Analysis**: Interpreting results and generating actionable insights
- **Custom Benchmarks**: Building your own evaluation tasks

### Next Steps:
1. **Run the Full Scripts**: Execute `benchmark_runner.py` and `enhanced_benchmark_runner.py`
2. **Experiment with Different Models**: Try GPT, T5, or other transformer models
3. **Create Domain-Specific Benchmarks**: Build evaluations for your specific use case
4. **Scale Up**: Use larger datasets and more comprehensive evaluation suites
5. **Deploy Models**: Use benchmark results to select models for production

### Resources:
- 📚 [Hugging Face Transformers Documentation](https://huggingface.co/transformers/)
- 🔬 [Papers With Code - NLP Benchmarks](https://paperswithcode.com/area/natural-language-processing)
- 🛠️ [DeepEval Framework](https://github.com/confident-ai/deepeval)
- 📊 [Weights & Biases for Experiment Tracking](https://wandb.ai/)
